# Audit dataset SIBI dan BISINDO
Secara default notebook membaca hasil audit tersimpan. Ubah flag di bawah menjadi True untuk menjalankan ulang. Dataset mentah tidak diubah. Dependensi dasar ada di requirements-audit.txt; uji landmark memerlukan requirements-landmark-audit.txt. Gunakan kernel Python lingkungan tersebut.

Audit struktur membaca seluruh gambar dan dapat memakan beberapa menit. Uji landmark mengambil 20 gambar per kelas, seed 42; hasilnya bukan akurasi classifier.

Git menyimpan notebook sumber tanpa output gambar. Hasil eksekusi lokal dapat disimpan di notebooks/executed/ (diabaikan Git). Setelah clone, ringkasan tersedia; jalankan audit dengan dataset lokal untuk membuat laporan detail dan contoh gambar.

In [ ]:
from pathlib import Path
import csv, json, subprocess, sys
from html import escape
from IPython.display import display, Image, Markdown, HTML
root = Path.cwd()
if not (root / 'scripts/audit_dataset.py').exists():
    root = root.parent
assert (root / 'scripts/audit_dataset.py').exists(), 'Jalankan notebook dari root proyek atau folder notebooks.'
out = root / 'reports/dataset_audit'
RUN_FULL_AUDIT = False
RUN_LANDMARK_SAMPLE = False
if RUN_FULL_AUDIT:
    subprocess.run([sys.executable, str(root / 'scripts/audit_dataset.py')], cwd=root, check=True)
if RUN_LANDMARK_SAMPLE:
    subprocess.run([sys.executable, str(root / 'scripts/audit_landmarks.py'), '--per-class', '20', '--seed', '42'], cwd=root, check=True)
print('Folder laporan:', out)

def show_local_image(path):
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"Gambar lokal belum tersedia: {path.name}. Jalankan audit terkait untuk membuatnya.")

## Jumlah gambar dan kelas
Kelas SIBI diturunkan dari nama file dan diperiksa terhadap ID anotasi; pemetaan resmi masih perlu dikonfirmasi.

In [ ]:
summary = json.loads((out / 'summary.json').read_text(encoding='utf-8'))
table = ['| Varian | Gambar | Kelas | Gambar rusak | Masalah format/anotasi kosong |', '|---|---:|---:|---:|---:|']
for name, row in summary['variants'].items():
    table.append(f"| {name} | {row['images']} | {len(row['classes'])} | {row['corrupt']} | {row['annotation_errors']} |")
display(Markdown('\n'.join(table)))
print('Label tanpa pasangan gambar:', len(summary['orphan_labels']))

## Kasus untuk tinjauan label
Daftar ini menandai kandidat masalah. Label tidak dikoreksi otomatis. ID mayoritas adalah hasil inferensi dari nama file, bukan konfigurasi resmi.

In [ ]:
review = json.loads((out / 'review_summary.json').read_text(encoding='utf-8'))
print('Gambar perlu ditinjau:', review['review_images'])
print('Nama huruf vs ID anotasi berbeda:', review['annotation_disagreements'])
with (out / 'review_required.csv').open(encoding='utf-8', newline='') as f:
    issues = list(csv.DictReader(f))
columns = ['filename_letter', 'annotation_ids', 'majority_id_for_filename_letter', 'reason', 'path']
html = '<table><tr>' + ''.join('<th>' + escape(c) + '</th>' for c in columns) + '</tr>'
for row in issues:
    html += '<tr>' + ''.join('<td>' + escape(row[c]) + '</td>' for c in columns) + '</tr>'
display(HTML(html + '</table>'))
show_local_image(out / 'annotation_review_samples.jpg')

## Potensi kebocoran sumber gambar
Hash identik hanya menemukan duplikasi file. Penggantian latar mengubah hash meskipun sumber tangan sama. Kandidat sumber ditentukan dari nama file setelah awalan bg_XX dihapus; verifikasi sebelum membuat split baru.

In [ ]:
print('Duplikasi hash:', summary['duplicate_summary']['sha256'])
print('Kandidat sumber lintas split per varian:', review['candidate_groups_by_variant'])
show_local_image(out / 'background_source_candidate_sample.jpg')

## Keberhasilan ekstraksi landmark pada sampel
Minimal satu tangan terdeteksi tidak memastikan pose dua tangan lengkap, landmark benar, atau huruf dikenali. Pengujian memakai API legacy MediaPipe 0.10.21, bukan benchmark webcam. Lihat konfigurasi lengkap di landmark_summary.json.

In [ ]:
landmark_path = out / 'landmark_summary.json'
if landmark_path.exists():
    landmarks = json.loads(landmark_path.read_text(encoding='utf-8'))
    table = ['| Dataset | Sampel | 0 tangan | 1 tangan | 2 tangan | Minimal 1 tangan |', '|---|---:|---:|---:|---:|---:|']
    for name, row in landmarks['totals'].items():
        counts = row['hands_detected']
        table.append(f"| {name} | {row['sampled']} | {counts.get('0', 0)} | {counts.get('1', 0)} | {counts.get('2', 0)} | {row['at_least_one_rate']:.1%} |")
    display(Markdown('\n'.join(table)))
else:
    print('Aktifkan RUN_LANDMARK_SAMPLE untuk membuat laporan ini.')

## Contoh visual
Lembar contoh awal memilih satu gambar per kelas, bukan sampel representatif seluruh pengguna/sesi. Lembar no_hand menampilkan contoh kegagalan detektor.

In [ ]:
for path in sorted(out.glob('*_samples.jpg')):
    if path.name == 'annotation_review_samples.jpg':
        continue
    print(path.name)
    display(Image(filename=str(path)))

## Interpretasi dan tindak lanjut
Baca [FINDINGS.md](../reports/dataset_audit/FINDINGS.md) untuk interpretasi hasil pada 13 Oktober 2026. Laporan tersebut adalah snapshot tertulis dan perlu ditinjau kembali setelah dataset atau konfigurasi audit berubah. Audit ini belum memverifikasi kebenaran linguistik seluruh pose, kemiripan visual menyeluruh, identitas pengguna, maupun performa classifier pada webcam.